# MMS Poisson — Method of Manufactured Solutions

**Poisson equation on the unit square**: −Δu = f, with weak penalty BC.

**Manufactured solution**: u = sin(πx)sin(πy) + xy (non-zero boundary data).

**Scheme**: P1 Lagrange, weak-penalty BC γ = C/h, residual form + Newton,
structured triangular meshes N×N, N = 8, 16, 32, 64, 128.

**Reference rates**: L2 → 2, H1 → 1 (optimal for P1).

In [ ]:
# --- canonical: bootstrap v1 ---
import sys, os, subprocess

_on_colab = "google.colab" in sys.modules

if not _on_colab:
    os.environ.setdefault("OMP_NUM_THREADS", "1")

if _on_colab:
    try:
        import gmsh
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/gmsh-install.sh"'
            ' -O /tmp/gmsh-install.sh && bash /tmp/gmsh-install.sh',
            shell=True, check=True,
        )
    try:
        import dolfinx
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh"'
            ' -O /tmp/fenicsx-install.sh && bash /tmp/fenicsx-install.sh',
            shell=True, check=True,
        )

import dolfinx
print("dolfinx version:", dolfinx.__version__)

if _on_colab:
    from google.colab import files
# --- end canonical: bootstrap ---

In [ ]:
import os, math
import numpy as np
from mpi4py import MPI
import ufl
from petsc4py import PETSc
from dolfinx.mesh import create_unit_square, CellType
from dolfinx.fem import (
    functionspace, Function, Constant, form, assemble_scalar,
)
from dolfinx.fem.petsc import NonlinearProblem
import basix.ufl as bufl


In [ ]:
# DD2365_FAST: run N=8,16 only for CI
import os
N_list = [8, 16] if os.environ.get('DD2365_FAST') == '1' else [8, 16, 32, 64, 128]
C_penalty = 1e3


## Manufactured solution

u_ex = sin(πx)sin(πy) + xy  →  f = −Δu_ex = 2π²sin(πx)sin(πy)

Boundary data g = u_ex|∂Ω: zero at x=0 and y=0, y at x=1, x at y=1.

In [ ]:
def u_ex_fn(x):
    return np.sin(np.pi * x[0]) * np.sin(np.pi * x[1]) + x[0] * x[1]

def f_fn(x):
    return 2.0 * np.pi**2 * np.sin(np.pi * x[0]) * np.sin(np.pi * x[1])

# Boundary function g = u_ex (same lambda reused)
g_fn = u_ex_fn


## Convergence study  (C = {C_penalty})

Poisson residual form:

    F(u; v) = ∫∇u·∇v dx + γ∫(u − g)v ds − ∫f·v dx = 0,   γ = C/h

Solved with NonlinearProblem + NewtonSolver (converges in 1 step — linear problem).

In [ ]:
def run_poisson(N_list, C):
    """Run Poisson MMS convergence study and return list of result dicts."""
    results = []
    for N in N_list:
        msh = create_unit_square(MPI.COMM_WORLD, N, N, cell_type=CellType.triangle)
        P1  = bufl.element('Lagrange', msh.basix_cell(), 1)
        V   = functionspace(msh, P1)

        u   = Function(V)
        v   = ufl.TestFunction(V)

        h_c   = ufl.CellDiameter(msh)
        gamma = Constant(msh, PETSc.ScalarType(C)) / h_c

        # Interpolate g and f into P1 functions
        g_h = Function(V)
        g_h.interpolate(g_fn)
        f_h = Function(V)
        f_h.interpolate(f_fn)

        # Residual: ∫∇u·∇v dx + γ(u−g)v ds − fv dx
        F = (
            ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx
            + gamma * ufl.inner(u - g_h, v) * ufl.ds
            - ufl.inner(f_h, v) * ufl.dx
        )

        problem = NonlinearProblem(
            F, u,
            petsc_options_prefix='mms_poisson_',
            petsc_options={
                'snes_type': 'newtonls',
                'snes_rtol': 1e-12,
                'snes_atol': 1e-14,
                'ksp_type': 'preonly',
                'pc_type': 'lu',
                'pc_factor_mat_solver_type': 'mumps',
            },
        )
        problem.solve()
        n_iters = problem.solver.getIterationNumber()

        # Error functions
        u_ex_fn_h = Function(V)
        u_ex_fn_h.interpolate(u_ex_fn)
        err = Function(V)
        err.x.array[:] = u.x.array - u_ex_fn_h.x.array

        e_L2 = math.sqrt(float(msh.comm.allreduce(
            assemble_scalar(form(err**2 * ufl.dx)), op=MPI.SUM)))
        e_H1 = math.sqrt(float(msh.comm.allreduce(
            assemble_scalar(form(ufl.inner(ufl.grad(err), ufl.grad(err)) * ufl.dx)),
            op=MPI.SUM)))

        h_val = 1.0 / N
        results.append({'N': N, 'h': h_val, 'e_L2': e_L2, 'e_H1': e_H1,
                         'iters': n_iters})
        print(f'N={N:4d}  h={h_val:.4f}  e_L2={e_L2:.4e}  e_H1={e_H1:.4e}  iters={n_iters}')
    return results


In [ ]:
print(f'C = {C_penalty:.0e}')
results_default = run_poisson(N_list, C_penalty)


### Results table (C = {C_penalty})

In [ ]:
def print_table(results, label=''):
    if label:
        print(f'--- {label} ---')
    print(f"{'N':>6}  {'h':>8}  {'e_L2':>12}  {'rate_L2':>8}  {'e_H1':>12}  {'rate_H1':>8}")
    for i, r in enumerate(results):
        if i == 0:
            r2 = r1 = '   —'
        else:
            prev = results[i-1]
            r2 = f'{math.log2(prev["e_L2"]/r["e_L2"]):8.2f}' if r['e_L2'] > 0 else '  NaN'
            r1 = f'{math.log2(prev["e_H1"]/r["e_H1"]):8.2f}' if r['e_H1'] > 0 else '  NaN'
        print(f"{r['N']:>6}  {r['h']:>8.4f}  {r['e_L2']:>12.4e}  {r2}  {r['e_H1']:>12.4e}  {r1}")

print_table(results_default, f'C = {C_penalty:.0e}')


### Penalty sensitivity

Re-run at N = {N_list} with C = 1e1 (under-penalized) and C = 1e5 (over-penalized).

In [ ]:
print('\n=== C = 1e1 ===')
results_c1 = run_poisson(N_list, 1e1)
print('\n=== C = 1e5 ===')
results_c5 = run_poisson(N_list, 1e5)


In [ ]:
print_table(results_c1, 'C = 1e1')
print()
print_table(results_c5, 'C = 1e5')
